# 9. Merge overlapping coverage periods

**Track:** sql | **Difficulty:** hard | **Tags:** gaps and islands, date ranges, window functions, running max

Run the three setup cells below once, in order. Then read the problem, write your
answer in the answer cell so that it ends with a DataFrame called `result`, and run
`check(result)`. You can run the check as many times as you like.

Free Colab runs only two or three notebooks at a time. When you are done here, end the
session with Runtime > Disconnect and delete runtime before opening the next problem.

In [ ]:
# Setup: installs PySpark on Colab and starts a local Spark session (about a minute).
import os
import re
import shutil
import subprocess
import sys

PYSPARK_VERSION = "4.1.1"

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"pyspark=={PYSPARK_VERSION}"],
        check=True,
    )
    # PySpark 4 needs Java 17 or newer.
    java_major = 0
    if shutil.which("java"):
        version_text = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
        match = re.search(r'version "(\d+)', version_text)
        java_major = int(match.group(1)) if match else 0
    if java_major < 17:
        subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-17-jdk-headless"], check=True)
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder.master("local[2]")
    .appName("009-merge-overlapping-periods")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "is ready.")

In [ ]:
#@title Load the data (run this cell, no need to read it)
import base64
import pathlib

_DATA = {
    "coverage_periods": """
UEFSMRUEFeADFY4CTBU8FQASAADwAQj7AQAFAQQQAgUHBAAFDQgA/w0YAP4NCAAHDRgADQ0IAAANCAAKDQgAAg0IAAsNCAASDQgA
9g1AAAMNEAAGDQgAEQ0IAA8NCAAJDQgA9w0wAA4NEAD9DRAABA0QAAgNCAABDQgA+A0gAPwNCAAMDRgA+g0QPPkBAAAAAAAA9QEA
AAAAAAAVABWAARWGASwVogEVEBUGFQYcGAgSAgAAAAAAABgI9QEAAAAAAAAWACgIEgIAAAAAAAAYCPUBAAAAAAAAEREAAABA8D8D
AAAAogEBBRcgiEEKEOagNJRa4bDmzoMiSjQpSa7Uc6qVz4lzlcP5Fke0bdbBlJNpyUAh9+O735fgYwIAAAAAFQQV+AMVgARMFX4V
ABIAAPwB8PsqTQAAQ00AAGBNAAALTQAAWk0AAGhNAAAATgAA600AADFNAADYTQAAK00AAKNNAADFTQAA4E0AABNNAABATQAAGU0A
AMFNAAAsTgAAPU4AADdOAABmTQAARU0AAFVNAAA5TQAAP00AABFNAAB0TQAAqk0AAGFNAABHTQAAD00AAIhNAABtTgAAUU0AAFNN
AADPTQAAsk0AAPJNAACuTQAASE0AALFNAAANTQAAPk0AAMxNAAA0TQAAhE0AADpNAAAUTQAAI04AAHBNAAB3TQAAEk0AAP9NAACb
TQAAJU0AAIlNAAAzTQAA9U0AAC1NAABCTQAAO00AANBNAAAVABWWARWcASwVogEVEBUGFQYcGARtTgAAGAQLTQAAFgAoBG1OAAAY
BAtNAAAREQAAAEvwSgMAAACiAQEGF0AgDEQxGMKBJMrCNM4DRdJEVYF1YZm2cZ33gaE4DmRpnmgaqOvjsYjVui8MxPIsuC1d2/CN
I2i+6XvO9z4AAAAAABUEFcgEFdIETBWSARUAEgAApAL0IwFFTQAAa00AAHRNAAApTQAAgU0AAHlNAAAKTgAAaE0AAAJOAAA/TQAA
600AAD1NAAC2TQAA300AAP5NAABCTQAAbE0AAE1NAADPTQAAU04AAFBOAABDTgAAb00AAGVNAAByTQAAg00AAGBNAAAuTQAAfk0A
ANpNAACMTQAAUE0AADhNAACxTQAAg04AAGNNAAB4TQAAAU4AAMdNAAD/TQAAtU0AAHdNAADKTQAAbk0AABdNAABPTQAAak0AAPxN
AABpTQAAcE0AAEBNAACiTQAAXE0AACNNAADATQAAPE4AAIZNAACoTQAASk0AAExNAAAcTgAAsE0AAB5NAABGTQAAvk0AAFlNAACA
TQAAOU0AAJ9NAAAWTgAAQU0AAGJNAAD6TQAAFQAVrAEVsgEsFaIBFRAVBhUGHBgEg04AABgEF00AABYAKASDTgAAGAQXTQAAEREA
AABW8FUDAAAAogEBBxeAgGBAKAwMB0RCsWA0HA9IRDKhVCwXTEYj2HA6ng8oJBqRLiXTCZVSrViWlusFi8lmtJrthsu5Zrodr+f7
AYPCIbFoBB0BSAAAAAAAABUEGUw1ABgGc2NoZW1hFQYAFQQlAhgHdXNlcl9pZAAVAiUCGApzdGFydF9kYXRlJQxMbAAAABUCJQIY
CGVuZF9kYXRlJQxMbAAAABaiARkcGTwmABwVBBk1AAYQGRgHdXNlcl9pZBUCFqIBFoQGFrgEJrYCJggcGAgSAgAAAAAAABgI9QEA
AAAAAAAWACgIEgIAAAAAAAAYCPUBAAAAAAAAEREAGSwVBBUAFQIAFQAVEBUCADwpBhkmAKIBAAAAJgAcFQIZNQAGEBkYCnN0YXJ0
X2RhdGUVAhaiARaSBhagBibgCCbABBwYBG1OAAAYBAtNAAAWACgEbU4AABgEC00AABERABksFQQVABUCABUAFRAVAgA8KQYZJgCi
AQAAACYAHBUCGTUABhAZGAhlbmRfZGF0ZRUCFqIBFvoGFooHJtQPJuAKHBgEg04AABgEF00AABYAKASDTgAAGAQXTQAAEREAGSwV
BBUAFQIAFQAVEBUCADwpBhkmAKIBAAAAFpATFqIBJggW4hEAGRwYDEFSUk9XOnNjaGVtYRjMAi8vLy8vL0FBQUFBUUFBQUFBQUFL
QUF3QUJnQUZBQWdBQ2dBQUFBQUJCQUFNQUFBQUNBQUlBQUFBQkFBSUFBQUFCQUFBQUFNQUFBQ0VBQUFBT0FBQUFBUUFBQUNZLy8v
L0FBQUJDQkFBQUFBY0FBQUFCQUFBQUFBQUFBQUlBQUFBWlc1a1gyUmhkR1VBQUFBQXp2Ly8vd0FBQUFESS8vLy9BQUFCQ0JBQUFB
QWtBQUFBQkFBQUFBQUFBQUFLQUFBQWMzUmhjblJmWkdGMFpRQUFBQUFHQUFnQUJnQUdBQUFBQUFBQUFCQUFGQUFJQUFZQUJ3QU1B
QUFBRUFBUUFBQUFBQUFCQWhBQUFBQWdBQUFBQkFBQUFBQUFBQUFIQUFBQWRYTmxjbDlwWkFBSUFBd0FDQUFIQUFnQUFBQUFBQUFC
UUFBQUFBQUFBQUE9ABggcGFycXVldC1jcHAtYXJyb3cgdmVyc2lvbiAyNS4wLjEZPBwAABwAABwAAAAZAwAAUEFSMQ==
""",
}

_data_dir = pathlib.Path("data") / "009-merge-overlapping-periods"
_data_dir.mkdir(parents=True, exist_ok=True)
for _name, _encoded in _DATA.items():
    (_data_dir / f"{_name}.parquet").write_bytes(base64.b64decode(_encoded))

coverage_periods = spark.read.parquet((_data_dir / "coverage_periods.parquet").resolve().as_posix())
coverage_periods.createOrReplaceTempView("coverage_periods")

for _name in _DATA:
    print(f"{_name}: {spark.table(_name).count()} rows")

In [ ]:
#@title Grader (run this cell, no need to read it)
"""Result grading shared by the build script and every generated notebook.

build_problem.py pastes this file into the notebook verbatim, so it must stay
standalone: standard library only, no imports from the rest of the toolkit.

A result is reduced to a "summary": its columns and types, its row count and
SHA-256 fingerprints of its normalized rows. Only the summary of the correct
answer is shipped in a notebook, never the rows themselves.
"""
import datetime
import decimal
import hashlib
import json
import math

FLOAT_DECIMALS = 6


def normalize_value(value):
    """Turn one cell value into something json.dumps renders the same way everywhere."""
    if value is None or isinstance(value, (bool, int, str)):
        return value
    if isinstance(value, float):
        if math.isnan(value):
            return "NaN"
        if math.isinf(value):
            return "Infinity" if value > 0 else "-Infinity"
        return round(value, FLOAT_DECIMALS) + 0.0  # + 0.0 turns -0.0 into 0.0
    if isinstance(value, decimal.Decimal):
        return format(value, "f")
    if isinstance(value, (datetime.datetime, datetime.date)):
        return value.isoformat()
    if isinstance(value, (bytes, bytearray)):
        return bytes(value).hex()
    if hasattr(value, "asDict"):  # pyspark Row (struct column)
        return {key: normalize_value(item) for key, item in value.asDict().items()}
    if isinstance(value, dict):  # map column
        pairs = [[normalize_value(key), normalize_value(item)] for key, item in value.items()]
        return sorted(pairs, key=_encode)
    if isinstance(value, (list, tuple)):
        return [normalize_value(item) for item in value]
    return str(value)


def _encode(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"))


def _digest(lines):
    sha = hashlib.sha256()
    for line in lines:
        sha.update(line.encode("utf-8"))
        sha.update(b"\n")
    return sha.hexdigest()


def describe_schema(df):
    """Column names and Spark types, sorted by name so column order is ignored."""
    return sorted([field.name, field.dataType.simpleString()] for field in df.schema.fields)


def summarize_rows(columns, rows, order_matters=False):
    """columns: sorted [name, type] pairs. rows: normalized values in that column order."""
    encoded = [_encode(row) for row in rows]
    if not order_matters:
        encoded.sort()
    column_fingerprints = {}
    for index, (name, _) in enumerate(columns):
        column_fingerprints[name] = _digest(sorted(_encode(row[index]) for row in rows))
    return {
        "columns": [list(column) for column in columns],
        "row_count": len(rows),
        "order_matters": order_matters,
        "fingerprint": _digest([_encode(columns)] + encoded),
        "column_fingerprints": column_fingerprints,
    }


def summarize(df, order_matters=False):
    columns = describe_schema(df)
    names = [name for name, _ in columns]
    # collect() returns `timestamp` values in the machine's local time zone, which differs
    # between the build machine and Colab. Casting to string inside Spark uses the session
    # time zone instead (UTC in both places). `timestamp_ntz` columns are not affected.
    expressions = [
        "CAST(`%s` AS STRING) AS `%s`" % (name, name) if kind == "timestamp" else "`%s`" % name
        for name, kind in columns
    ]
    collected = df.selectExpr(*expressions).collect()
    rows = [[normalize_value(row[name]) for name in names] for row in collected]
    return summarize_rows(columns, rows, order_matters)


def grade(df, expected):
    """Compare a DataFrame to the expected summary. Returns (passed, hints).

    The hints say what kind of thing is wrong (columns, types, row count, which
    column) but never what the right values are.
    """
    if not hasattr(df, "schema") or not hasattr(df, "collect"):
        return False, ["check() needs a Spark DataFrame, got %s." % type(df).__name__]

    names = [field.name for field in df.schema.fields]
    duplicates = sorted({name for name in names if names.count(name) > 1})
    if duplicates:
        return False, ["Your result has duplicate column names: %s." % ", ".join(duplicates)]

    actual = dict(describe_schema(df))
    wanted = dict(expected["columns"])
    hints = []
    missing = sorted(set(wanted) - set(actual))
    extra = sorted(set(actual) - set(wanted))
    if missing:
        hints.append("Missing columns: %s." % ", ".join(missing))
    if extra:
        hints.append("Unexpected columns: %s." % ", ".join(extra))
    for name in sorted(set(wanted) & set(actual)):
        if wanted[name] != actual[name]:
            hints.append("Column %s has type %s, expected %s." % (name, actual[name], wanted[name]))
    if hints:
        return False, hints

    row_count = df.count()
    if row_count != expected["row_count"]:
        return False, [
            "Row count is wrong: your result has %d rows, expected %d."
            % (row_count, expected["row_count"])
        ]

    summary = summarize(df, expected["order_matters"])
    if summary["fingerprint"] == expected["fingerprint"]:
        return True, []

    wrong = sorted(
        name
        for name, fingerprint in summary["column_fingerprints"].items()
        if fingerprint != expected["column_fingerprints"][name]
    )
    if wrong:
        return False, ["Columns and row count are right, but values are wrong in: %s." % ", ".join(wrong)]
    if expected["order_matters"]:
        return False, ["The values are right, but the rows are in the wrong order."]
    return False, ["Each column has the right set of values, but they are combined into the wrong rows."]


def make_check(expected):
    def check(df):
        passed, hints = grade(df, expected)
        if passed:
            print("Correct. Your result matches the expected output.")
        else:
            print("Not correct yet.")
            for hint in hints:
                print("  - " + hint)
        # Returns nothing, so a notebook cell ending in check(result) prints only the message.

    return check

EXPECTED = {'column_fingerprints': {'end_date': 'b8e92bf85e869ff7f28f2a049fba0877f054a27bf25b143da28be905bdcfcae6',
                         'start_date': 'f15402d70df2e1d53e18808f78e302e0e99f7bd23d369ee80bcd28a297ae4455',
                         'user_id': 'd6a1f7e14fa8755991cae12d3b96ec5455939f981ff7c6f1553870ef4217dbb4'},
 'columns': [['end_date', 'date'], ['start_date', 'date'], ['user_id', 'bigint']],
 'fingerprint': '5f0d26ee881e0028117031e262b30f4656357ccb506ef07d2c5e530dcfa8ed1b',
 'order_matters': False,
 'row_count': 52}

check = make_check(EXPECTED)

## The situation

Each row of `coverage_periods` says a user had an active plan from one date to
another. Renewals, upgrades and re-sent records mean the periods of one user overlap,
sit inside each other, or follow each other without a break.

Billing needs the continuous stretches: from when to when was the user covered
without interruption?

## Input tables

### `coverage_periods`

One row per recorded period. A user has one or more rows.

| Column | Type | Description |
|---|---|---|
| `user_id` | BIGINT | The user. |
| `start_date` | DATE | First covered day. |
| `end_date` | DATE | Last covered day (inclusive). Never before `start_date`. |

## Requirement

Merge each user's periods into continuous blocks and return one row per block.

- Two periods belong to the same block if they overlap, or if one starts the day
  after the other ends. Covered up to 31 January and again from 1 February is
  continuous cover.
- One or more uncovered days between two periods separate them into different blocks.
- A period that lies completely inside another one changes nothing.
- A block's `start_date` is its first covered day and its `end_date` is its last
  covered day.

## Expected output

One row per continuous block.

| Column | Type | Description |
|---|---|---|
| `user_id` | BIGINT | |
| `start_date` | DATE | First day of the block. |
| `end_date` | DATE | Last day of the block. |

In [ ]:
coverage_periods.show(5)

## Your answer

Write one Spark SQL query. The tables are available as views: `coverage_periods`.

In [ ]:
query = """
-- Write your Spark SQL query here.
"""
result = spark.sql(query)
result.show()

In [ ]:
check(result)